# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    # (B, C, H, W) -> (B, N, P*P*C) with N = (H/P) * (W/P), patches in row-major (raster) order
    B, C, H, W = x.shape
    P = patch_size
    assert H % P == 0 and W % P == 0, "image size must be divisible by the patch size"
    x = x.reshape(B, C, H // P, P, W // P, P)      # split both spatial axes into (grid, within-patch)
    x = x.permute(0, 2, 4, 3, 5, 1)                 # (B, gh, gw, P, P, C): grid first, pixels last
    return x.reshape(B, (H // P) * (W // P), P * P * C)

# ---- a 28x28 MNIST image with P=4 becomes 7x7 = 49 tokens of 16 pixels each ----
img = torch.arange(28 * 28, dtype=torch.float32).reshape(1, 1, 28, 28)
tokens = patchify(img, 4)
print("image", tuple(img.shape), "-> tokens", tuple(tokens.shape))
print("token 0 is the top-left 4x4 block:", torch.equal(tokens[0, 0], img[0, 0, :4, :4].flatten()))
print("token 1 is the block to its right:", torch.equal(tokens[0, 1], img[0, 0, :4, 4:8].flatten()))
print("token 7 starts the second row of patches:", torch.equal(tokens[0, 7], img[0, 0, 4:8, :4].flatten()))
print("same as F.unfold (the conv-style patch extractor):",
      torch.equal(tokens, F.unfold(img, kernel_size=4, stride=4).transpose(1, 2)))

image (1, 1, 28, 28) -> tokens (1, 49, 16)
token 0 is the top-left 4x4 block: True
token 1 is the block to its right: True
token 7 starts the second row of patches: True
same as F.unfold (the conv-style patch extractor): True


In [3]:
# Patch projection + learnable 1D positional embedding, as in the ViT paper (Sec. 3.1, Eq. 1).
# The paper found learnable 1D embeddings as good as 2D-aware ones (Appendix D.4), so we use those.
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        # a shared linear map E applied to every flattened patch (equivalent to a PxP conv with stride P)
        self.proj = nn.Linear(patch_dim, d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        # (B, N, patch_dim) -> (B, N, d_model)
        return self.proj(x_patches)


class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        # one learnable vector per patch position; small init like the reference ViT implementation
        self.pos = nn.Parameter(torch.zeros(1, num_tokens, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # attention is permutation-invariant; adding E_pos tells the model WHERE each patch came from
        return x + self.pos          # (1, N, D) broadcasts over the batch

# ---- shapes through the embedding stage ----
pe, pos = PatchEmbed(16, 64), PositionalEmbedding(49, 64)
z = pos(pe(patchify(torch.rand(8, 1, 28, 28), 4)))
print("embedded tokens", tuple(z.shape), "  pos-embedding params", pos.pos.numel())

embedded tokens (8, 49, 64)   pos-embedding params 3136


In [4]:
# Variants compared (see the markdown cell below for the justification):
#   ffn_gelu  baseline     W2 · GELU(W1 x)
#   geglu     GLU variant  W2 · (GELU(W x) ⊙ V x)
#   swiglu    GLU variant  W2 · (Swish(W x) ⊙ V x)
# Following Shazeer (2020) all FFN projections are bias-free, and the gated hidden width is
# d_ff_gated = 2/3 · d_ff so that 3 · d · d_ff_gated ≈ 2 · d · d_ff (same parameters and FLOPs).
GLU_ACTIVATIONS = {
    "glu": torch.sigmoid,          # original GLU (Dauphin et al., 2016)
    "bilinear": lambda t: t,       # no nonlinearity at all
    "reglu": F.relu,
    "geglu": F.gelu,
    "swiglu": F.silu,              # Swish with beta = 1
}


class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff, bias=False)
        self.fc2 = nn.Linear(d_ff, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.drop(F.gelu(self.fc1(x)))
        return self.drop(self.fc2(h))


class GLUFeedForward(nn.Module):
    """GLU-family FFN"""
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        if variant not in GLU_ACTIVATIONS:
            raise ValueError(f"unknown GLU variant {variant!r}; choose from {sorted(GLU_ACTIVATIONS)}")
        self.variant = variant
        self.act = GLU_ACTIVATIONS[variant]
        self.w = nn.Linear(d_model, d_ff_gated, bias=False)    # gate branch a, goes through act
        self.v = nn.Linear(d_model, d_ff_gated, bias=False)    # value branch b, stays linear
        self.w2 = nn.Linear(d_ff_gated, d_model, bias=False)   # project back to d_model
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.drop(self.act(self.w(x)) * self.v(x))         # act(a) ⊙ b
        return self.drop(self.w2(h))

# ---- the 2/3 rule keeps the comparison parameter-matched ----
d, dff = 64, 256
count = lambda m: sum(p.numel() for p in m.parameters())
print(f"FFN  d_ff={dff}:            {count(FeedForward(d, dff, 0.0))} params")
for v in ["geglu", "swiglu"]:
    print(f"{v:6s} d_ff_gated={round(2 * dff / 3)}:  {count(GLUFeedForward(d, round(2 * dff / 3), 0.0, v))} params")
print("same width without the rule:", count(GLUFeedForward(d, dff, 0.0, "geglu")), "params (1.5x, an unfair comparison)")

FFN  d_ff=256:            32768 params
geglu  d_ff_gated=171:  32832 params
swiglu d_ff_gated=171:  32832 params
same width without the rule: 49152 params (1.5x, an unfair comparison)


### Which GLU variants and why

We compare the **GELU FFN baseline** against **GEGLU** and **SwiGLU**.

- **They are the two strongest variants in Shazeer (2020).** In Table 1 they reach the lowest pre-training perplexity of all variants tested (1.942 and 1.944 versus 1.997 for the GELU FFN). They also lead most GLUE and SuperGLUE columns (Tables 2 and 3). If gating helps at our tiny scale, these are the variants most likely to show it.
- **They are what modern models actually use.** SwiGLU is the FFN in PaLM and LLaMA. GEGLU is used in T5 v1.1 and Gemma. That makes the result relevant beyond MNIST.
- **They isolate the gating effect cleanly.** GEGLU uses the *same* activation as the baseline, so GELU FFN versus GEGLU measures only the effect of adding the gate. SwiGLU then varies the activation inside the gate.

**Fairness controls.**
- All FFNs are bias-free, as in the paper.
- The gated width follows the 2/3 rule, so all three variants have the same parameters and FLOPs, within 0.1%.
- Every variant sees the same hyper-parameters.
- For a given seed, every variant starts from the same RNG state and sees the **same mini-batch order**. The train loader has its own seeded generator, so seeds can be compared as *pairs*.

In [5]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        # batch_first=True -> inputs are (B, N, D) like the rest of the model
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = mlp
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)   # self-attention: q = k = v
        x = x + self.drop(attn_out)
        # our FFN modules already end with Dropout, which is the Dropout(MLP(...)) of the
        # docstring -- adding a second one here would drop the MLP output twice
        x = x + self.mlp(self.ln2(x))
        return x

# ---- a block maps (B, N, D) -> (B, N, D), so blocks can be stacked ----
blk = TransformerEncoderBlock(64, 4, FeedForward(64, 256, 0.1), 0.1)
print("(8, 49, 64) ->", tuple(blk(torch.randn(8, 49, 64)).shape))

(8, 49, 64) -> (8, 49, 64)


In [6]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size

        # patch projection + learnable positional embedding (+ embedding dropout as in ViT)
        self.patch_embed = PatchEmbed(patch_dim, d_model)
        self.pos_embed = PositionalEmbedding(self.num_tokens, d_model)
        self.emb_drop = nn.Dropout(dropout)

        # pick the MLP; a FRESH module per block, otherwise all blocks would share weights
        self.mlp_kind = mlp_kind
        def make_mlp() -> nn.Module:
            if mlp_kind == "ffn_gelu":
                return FeedForward(d_model, d_ff, dropout)
            if mlp_kind in GLU_ACTIVATIONS:
                return GLUFeedForward(d_model, round(2 * d_ff / 3), dropout, variant=mlp_kind)
            raise ValueError(f"unknown mlp_kind {mlp_kind!r}")

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=make_mlp(),
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        # pre-LN transformers need a final LayerNorm; then mean-pool tokens and classify
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = patchify(x, self.patch_size)                        # (B, N, P*P)
        h = self.emb_drop(self.pos_embed(self.patch_embed(tokens)))  # (B, N, D)
        for block in self.blocks:
            h = block(h)
        pooled = self.norm(h).mean(dim=1)                            # (B, D) mean pooling over tokens
        logits = self.head(pooled)                                   # (B, 10)
        return logits

# ---- end to end shape check + parameter counts of the three variants ----
for kind in ["ffn_gelu", "geglu", "swiglu"]:
    m = TinyViT(patch_size=4, d_model=64, n_heads=4, n_layers=2, d_ff=256, dropout=0.1, mlp_kind=kind)
    print(f"{kind:8s} logits {tuple(m(torch.rand(2, 1, 28, 28)).shape)}   trainable params {sum(p.numel() for p in m.parameters() if p.requires_grad)}")

ffn_gelu logits (2, 10)   trainable params 104330
geglu    logits (2, 10)   trainable params 104458
swiglu   logits (2, 10)   trainable params 104458


In [7]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cpu"  # set "cuda" if available

In [8]:
import time


def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    torch.manual_seed(cfg.seed)            # dropout masks depend only on the seed
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    test_accs: list[float] = []
    test_losses: list[float] = []
    epoch_train_losses: list[float] = []
    t_start = time.perf_counter()

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = F.cross_entropy(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())
        epoch_train_losses.append(sum(train_losses[-len(train_loader):]) / len(train_loader))

        # Evaluation loop NOTE: Should be no need to change this
        # (only addition: also accumulate the test loss, summed per example)
        model.eval()
        correct = 0.0
        total = 0.0
        loss_sum = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                loss_sum += F.cross_entropy(logits, yb, reduction="sum").item()
                total += yb.numel()

        test_accs.append(correct / total)
        test_losses.append(loss_sum / total)
        print(f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | train loss: {epoch_train_losses[-1]:.4f}"
              f" | test loss: {test_losses[-1]:.4f} | test acc: {test_accs[-1]:.4f}")

    # convergence speed: first step at which the 100-step moving average of the train loss drops below 0.2
    window, threshold = 100, 0.2
    steps_to_threshold = next(
        (i + 1 for i in range(window - 1, len(train_losses))
         if sum(train_losses[i - window + 1 : i + 1]) / window < threshold),
        None,
    )

    return {
        "mlp_kind": mlp_kind,
        "seed": cfg.seed,
        "n_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "train_losses": train_losses,              # per update step -> loss-vs-steps curves
        "epoch_train_losses": epoch_train_losses,  # mean train loss per epoch
        "test_losses": test_losses,                # per epoch
        "test_accs": test_accs,                    # per epoch
        "final_test_acc": test_accs[-1],
        "best_test_acc": max(test_accs),
        "best_epoch": test_accs.index(max(test_accs)) + 1,
        "steps_to_loss_0.2": steps_to_threshold,
        "train_time_s": time.perf_counter() - t_start,
    }

In [9]:
import dataclasses

cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01, device="cpu")
# CPU on purpose: CPU kernels are deterministic, so every number below is exactly reproducible.

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Tiny model example.
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

runs = ["ffn_gelu", "geglu", "swiglu"]   # baseline + two GLU variants (justified above)
seeds = [0, 1, 2, 3, 4]                  # 5 seeds per variant: differences on MNIST are small, so we need error bars
results = []

for seed in seeds:
    for kind in runs:
        run_cfg = dataclasses.replace(cfg, seed=seed)
        # same seed -> same RNG state at init and the same mini-batch order for every variant (paired design)
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0,
                                  generator=torch.Generator().manual_seed(seed))
        torch.manual_seed(seed)
        model = TinyViT(
            patch_size=patch_size,
            d_model=d_model,
            n_heads=n_heads,
            n_layers=n_layers,
            d_ff=d_ff,
            dropout=dropout,
            mlp_kind=kind,
        )
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\nRun: {kind} | seed {seed} | trainable params {n_params}")
        out = train_one_run(kind, model, train_loader, test_loader, run_cfg)
        print(f"   -> final acc {out['final_test_acc']:.4f}, best acc {out['best_test_acc']:.4f} "
              f"(epoch {out['best_epoch']}), {out['train_time_s']:.0f}s")
        results.append(out)


Run: ffn_gelu | seed 0 | trainable params 104330
[ffn_gelu] epoch 1/5 | train loss: 1.2689 | test loss: 0.5556 | test acc: 0.8196
[ffn_gelu] epoch 2/5 | train loss: 0.5102 | test loss: 0.3483 | test acc: 0.8913
[ffn_gelu] epoch 3/5 | train loss: 0.3524 | test loss: 0.2330 | test acc: 0.9303
[ffn_gelu] epoch 4/5 | train loss: 0.2748 | test loss: 0.2027 | test acc: 0.9378
[ffn_gelu] epoch 5/5 | train loss: 0.2279 | test loss: 0.1830 | test acc: 0.9456
   -> final acc 0.9456, best acc 0.9456 (epoch 5), 120s

Run: geglu | seed 0 | trainable params 104458
[geglu] epoch 1/5 | train loss: 1.2320 | test loss: 0.5328 | test acc: 0.8259
[geglu] epoch 2/5 | train loss: 0.4371 | test loss: 0.2799 | test acc: 0.9124
[geglu] epoch 3/5 | train loss: 0.2686 | test loss: 0.1863 | test acc: 0.9435
[geglu] epoch 4/5 | train loss: 0.1977 | test loss: 0.1284 | test acc: 0.9603
[geglu] epoch 5/5 | train loss: 0.1587 | test loss: 0.1245 | test acc: 0.9610
   -> final acc 0.9610, best acc 0.9610 (epoch 5), 1

In [10]:
# ---- summary over seeds: mean ± std, plus PAIRED comparisons against the baseline ----
def mean_std(xs):
    m = sum(xs) / len(xs)
    s = (sum((x - m) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5 if len(xs) > 1 else 0.0
    return m, s

def t_two_sided_p(t: float, df: int) -> float:
    """Two-sided p-value of Student's t via numerical integration of the density (no scipy needed)."""
    c = math.exp(math.lgamma((df + 1) / 2) - math.lgamma(df / 2)) / math.sqrt(df * math.pi)
    pdf = lambda u: c * (1 + u * u / df) ** (-(df + 1) / 2)
    a, n = abs(t), 2000                                   # Simpson's rule on [0, |t|]
    h = a / n
    area = pdf(0) + pdf(a) + sum((4 if k % 2 else 2) * pdf(k * h) for k in range(1, n))
    return max(0.0, 1 - 2 * area * h / 3)

by = {k: sorted([r for r in results if r["mlp_kind"] == k], key=lambda r: r["seed"]) for k in runs}

print(f"{len(seeds)} seeds per variant, {cfg.epochs} epochs each\n")
print(f"{'variant':9s} {'params':>7s} {'final acc (%)':>15s} {'best acc (%)':>14s} {'final test loss':>16s}"
      f" {'train loss ep1':>15s} {'steps to 0.2 (runs reaching)':>24s} {'time/run':>9s}")
for k in runs:
    rs = by[k]
    fa = mean_std([100 * r["final_test_acc"] for r in rs]); ba = mean_std([100 * r["best_test_acc"] for r in rs])
    tl = mean_std([r["test_losses"][-1] for r in rs]); e1 = mean_std([r["epoch_train_losses"][0] for r in rs])
    reached = [r["steps_to_loss_0.2"] for r in rs if r["steps_to_loss_0.2"] is not None]
    st = mean_std(reached) if reached else (float("nan"), float("nan")); tt = mean_std([r["train_time_s"] for r in rs])
    print(f"{k:9s} {rs[0]['n_params']:7d} {fa[0]:8.2f} ± {fa[1]:.2f} {ba[0]:7.2f} ± {ba[1]:.2f} {tl[0]:9.4f} ± {tl[1]:.4f}"
          f" {e1[0]:8.4f} ± {e1[1]:.4f} {st[0]:6.0f}±{st[1]:3.0f} ({len(reached)}/{len(rs)}) {tt[0]:6.0f}s")

print("\nTest accuracy per epoch (mean ± std over seeds, %):")
for k in runs:
    cols = [mean_std([100 * r["test_accs"][e] for r in by[k]]) for e in range(cfg.epochs)]
    print(f"  {k:9s} " + "  ".join(f"ep{e+1} {m:.2f}±{s:.2f}" for e, (m, s) in enumerate(cols)))

print("\nPer-seed final accuracy (%):")
for k in runs:
    print(f"  {k:9s} " + "  ".join(f"{100 * r['final_test_acc']:.2f}" for r in by[k]))

print("\nPaired difference vs ffn_gelu (same seed = same init RNG state and data order), two-sided paired t-test:")
for k in runs[1:]:
    for metric, scale, name in [("final_test_acc", 100, "final acc (pp)"), ("steps_to_loss_0.2", 1, "steps to 0.2")]:
        pairs = [(a[metric], b[metric]) for a, b in zip(by[k], by["ffn_gelu"]) if a[metric] is not None and b[metric] is not None]
        if len(pairs) < 2:
            print(f"  {k:7s} {name:15s} not enough runs reached the threshold for a paired test")
            continue
        d = [scale * (a - b) for a, b in pairs]
        m, s = mean_std(d)
        if s == 0:      # every seed gave exactly the same difference: a t-test is undefined
            print(f"  {k:7s} {name:15s} mean diff {m:+8.3f}  identical on every seed, no t-test")
            continue
        t = m / (s / math.sqrt(len(d)))
        print(f"  {k:7s} {name:15s} mean diff {m:+8.3f}  std {s:7.3f}  t = {t:+6.2f}  p = {t_two_sided_p(t, len(d) - 1):.3f}"
              f"   ({sum(x > 0 for x in d)}/{len(d)} seeds higher)")

5 seeds per variant, 5 epochs each

variant    params   final acc (%)   best acc (%)  final test loss  train loss ep1 steps to 0.2 (runs reaching)  time/run
ffn_gelu   104330    94.87 ± 0.56   94.87 ± 0.56    0.1662 ± 0.0193   1.2814 ± 0.0533   2129± 83 (2/5)    117s
geglu      104458    96.04 ± 0.38   96.04 ± 0.38    0.1335 ± 0.0118   1.1999 ± 0.0275   1809±152 (5/5)    113s
swiglu     104458    95.94 ± 0.38   95.94 ± 0.38    0.1342 ± 0.0130   1.2013 ± 0.0273   1842±167 (5/5)    104s

Test accuracy per epoch (mean ± std over seeds, %):
  ffn_gelu  ep1 81.11±1.51  ep2 89.67±0.33  ep3 92.94±0.32  ep4 94.23±0.49  ep5 94.87±0.56
  geglu     ep1 82.39±0.58  ep2 91.25±0.99  ep3 94.21±0.56  ep4 95.34±0.54  ep5 96.04±0.38
  swiglu    ep1 82.45±1.00  ep2 91.17±0.92  ep3 94.10±0.59  ep4 95.17±0.70  ep5 95.94±0.38

Per-seed final accuracy (%):
  ffn_gelu  94.56  95.04  95.76  94.33  94.68
  geglu     96.10  95.78  96.67  95.80  95.84
  swiglu    96.10  95.79  96.54  95.63  95.65

Paired differen

## Results and discussion

**Setup.** TinyViT with 2 layers, d_model 64, 49 patch tokens and about 104k parameters. Training uses AdamW at lr 3e-4 for 5 epochs. Each variant was run with **5 seeds**. Runs with the same seed share the initial RNG state and the mini-batch order, so the seeds are compared as pairs.

| variant | params | final test acc (%) | final test loss | train loss, epoch 1 | runs whose loss reached 0.2 |
|---|---|---|---|---|---|
| GELU FFN (baseline) | 104,330 | 94.87 ± 0.56 | 0.166 ± 0.019 | 1.281 ± 0.053 | 2 / 5 |
| GEGLU | 104,458 | **96.04 ± 0.38** | 0.134 ± 0.012 | 1.200 ± 0.028 | 5 / 5 |
| SwiGLU | 104,458 | 95.94 ± 0.38 | 0.134 ± 0.013 | 1.201 ± 0.027 | 5 / 5 |

All numbers are mean ± standard deviation over seeds. Best and final accuracy are identical because every run was still improving at epoch 5.

**1. Final accuracy: both GLU variants beat the baseline, and the effect is robust.**
- GEGLU is ahead by **+1.16 pp** and SwiGLU by **+1.07 pp**. Both win on **5 of 5 seeds**, with paired t-test p = 0.002 for each.
- The gain is two to three times larger than the seed-to-seed spread of about 0.4 to 0.6 pp. It also survives a Bonferroni correction for the two primary tests, whose threshold is 0.025.
- GEGLU against SwiGLU is a different story. The difference is +0.10 pp, with t = 2.27 and p = 0.086, which is not significant with 5 seeds. **We cannot tell the two gated variants apart.**

**2. Convergence speed: the GLU variants learn faster per step.**
- Their epoch-1 train loss is lower, and their test accuracy is ahead at **every** epoch (82.4 vs 81.1 % after epoch 1, 94.2 vs 92.9 % after epoch 3).
- Every GLU run pushed its 100-step average train loss below 0.2, after roughly 1,800 steps. Only 2 of 5 baseline runs got there within the 2,345-step budget. On the two seeds where the baseline did reach it, GLU got there about 350 steps earlier, roughly 17% sooner.

**3. Stability: no instabilities, and the GLU runs are slightly more consistent.**
- No run diverged.
- The spread of final accuracy is 0.38 pp for both GLU variants and 0.56 pp for the baseline. With 5 seeds that difference in spread is suggestive only, not established.

**Why might gating help?** The value branch is multiplied by a learned, input-dependent gate. The FFN can therefore compute multiplicative interactions between features that a plain FFN must approximate with more units. GEGLU uses the same GELU activation as the baseline, so the GELU FFN versus GEGLU comparison isolates **the gate itself** as the source of the gain. The choice of activation inside the gate, GELU or Swish, made no measurable difference. This matches Shazeer's finding that GEGLU and SwiGLU are close to each other and both ahead of the FFN.

**Caveats. These conclusions are about this budget and this scale.**
- **Fixed, short budget.** All models are still improving at epoch 5. The result is about *sample efficiency at a fixed number of steps*, so the gap may shrink with longer training.
- **One shared learning rate.** All variants used lr 3e-4 without per-variant tuning. A tuned baseline could close part of the gap.
- **Small study.** MNIST is easy, the models are tiny, and n = 5 seeds. The t-test assumes roughly normal paired differences, and a few more seeds would firm up the GEGLU versus SwiGLU and stability comparisons.
- **Same parameters, not quite the same cost.** The variants match in parameters and FLOPs through the 2/3 rule. GLU still runs three smaller matmuls instead of two, and wall-clock time per run was similar here at about 100 to 110 s on CPU.

**Reproducibility.**
- Seeds are fixed for initialization, dropout, and data order.
- The train loader has its own `torch.Generator`, so data order does not depend on how much randomness the model consumed.
- Everything runs on CPU, whose kernels are deterministic.
- **Verified:** re-executing the notebook code from scratch reproduced every per-epoch train loss, test loss and test accuracy of all 15 runs exactly.